# Unidad 3 · Prototipo de detección y priorización de fraude

**Universidad Peruana Unión · Inteligencia Artificial · Glender Vargas Chipana**

Este notebook compara la **MLP de las clases 14 y 15**, adaptada de la predicción de abandono a clasificación de fraude, con **HistGradientBoosting**, la arquitectura seleccionada por el equipo en la Unidad 2. Ambos modelos usan IBM TabFormer. La demo genera una puntuación, una prioridad y una recomendación para el analista.

Ejecuta las celdas desde el repositorio con las dependencias de `unidad3/requirements.txt`. Los modelos ya están entrenados.

In [1]:
from pathlib import Path
import sys
import pandas as pd

inicio = Path.cwd()
raiz = next((p for p in (inicio, *inicio.parents) if (p / "unidad3" / "modelos.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Abre el notebook desde una copia del repositorio.")
sys.path.insert(0, str(raiz))

from unidad3.datos import MODALIDADES
from unidad3.modelos import HGB, MLP, Prototipo

prototipo = Prototipo()
compra_prioritaria = ["2019-06-17", "12:30", 850.00, "Chip Transaction", "CA", 5411]
compra_online = ["2019-06-15", "02:15", 850.00, "Online Transaction", "ONLINE", 5812]
compra_presencial = ["2019-06-17", "14:30", 35.50, "Chip Transaction", "CA", 5411]
devolucion = ["2019-06-17", "14:30", -35.50, "Chip Transaction", "CA", 5411]

def mostrar(modelo, casos):
    filas = []
    for nombre, datos in casos:
        r = prototipo.evaluar(modelo, *datos)
        p = "Fuera de cohorte" if r["puntuacion"] is None else f"{r['puntuacion'] * 100:.2f}/100"
        filas.append({"Caso": nombre, "Puntuación": p, "Prioridad": r["prioridad"], "Acción": r["accion"]})
    return pd.DataFrame(filas)


## 1. Red neuronal MLP de la clase

**26 entradas → Dense 64 (ReLU) → BatchNormalization → Dropout 0,30 → Dense 32 (ReLU, L2 = 0,01) → Dropout 0,20 → Dense 1 (sigmoide).**

El modelo usa entropía cruzada binaria ponderada, Adam con tasa de aprendizaje 0,001 y lotes de 32. La parada temprana usa la pérdida ponderada del bloque de validación, paciencia 10 y recuperación de los mejores pesos. La arquitectura conserva las dos capas ocultas y la regularización del ejemplo del docente. La dimensión de entrada cambia para admitir las variables de TabFormer transformadas por el preprocesador.

In [2]:
casos = [("Compra prioritaria", compra_prioritaria), ("Compra en línea", compra_online), ("Compra presencial", compra_presencial), ("Importe negativo", devolucion)]
mostrar(MLP, casos)

,Caso,Puntuación,Prioridad,Acción
0,Compra prioritaria,90.22/100,Revisión prioritaria,Enviar a un analista para verificar la operación.
1,Compra en línea,1.37/100,Seguimiento habitual,Continuar el flujo habitual y conservar el reg...
2,Compra presencial,78.96/100,Seguimiento habitual,Continuar el flujo habitual y conservar el reg...
3,Importe negativo,Fuera de cohorte,Fuera del alcance,"Derivar a revisión de reembolsos, contracargos..."


## 2. HistGradientBoosting del equipo

**26 entradas → ensamble aditivo de árboles entrenados por gradientes → puntuación de fraude.**

Se usa `log_loss` con pesos de clase balanceados. La tasa de aprendizaje es 0,10, con 100 iteraciones, hasta 31 hojas por árbol y al menos 20 observaciones por hoja. El reentrenamiento de la Unidad 3 usa el mismo bloque de ajuste que la MLP y 100 iteraciones fijas.

In [3]:
mostrar(HGB, casos)

,Caso,Puntuación,Prioridad,Acción
0,Compra prioritaria,96.52/100,Revisión prioritaria,Enviar a un analista para verificar la operación.
1,Compra en línea,0.03/100,Seguimiento habitual,Continuar el flujo habitual y conservar el reg...
2,Compra presencial,76.81/100,Seguimiento habitual,Continuar el flujo habitual y conservar el reg...
3,Importe negativo,Fuera de cohorte,Fuera del alcance,"Derivar a revisión de reembolsos, contracargos..."


## 3. Evaluación temporal

Ajuste de noviembre de 2017 a octubre de 2018, validación durante noviembre y diciembre de 2018 y prueba durante 2019. El umbral por modelo se fijó para priorizar cerca del 5 % del bloque de validación. La decisión usa el puntaje sin redondear.

In [4]:
filas = []
for nombre in (MLP, HGB):
    r = prototipo.metricas["modelos"][nombre]
    op = r["test_operativo"]
    filas.append({
        "Modelo": nombre,
        "Average Precision": f"{r['test_average_precision']:.4f}",
        "ROC-AUC": f"{r['test_roc_auc']:.4f}",
        "Alertas": op["alertas"],
        "Fraudes detectados": op["confusion"]["tp"],
        "Precisión": f"{op['precision']:.2%}",
        "Recall": f"{op['recall']:.2%}",
    })
pd.DataFrame(filas)

,Modelo,Average Precision,ROC-AUC,Alertas,Fraudes detectados,Precisión,Recall
0,MLP,0.0099,0.8402,81855,718,0.88%,35.32%
1,HistGradientBoosting,0.0123,0.8333,81673,596,0.73%,29.32%


HistGradientBoosting obtuvo la mayor Average Precision en validación y prueba. Con una tasa de alertas cercana al 5 % en 2019, la MLP recuperó más fraudes. Ambos modelos presentan una precisión de alertas inferior al 1 % para este punto operativo. Las alertas requieren revisión humana.

## 4. Formulario interactivo

Cambia el modelo o edita los campos y ejecuta la evaluación. Los ejemplos de la demo se diseñaron para probar inferencia y la ruta de excepción.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

modelo = widgets.Dropdown(options=[MLP, HGB], description="Modelo")
fecha = widgets.Text(value=compra_online[0], description="Fecha")
hora = widgets.Text(value=compra_online[1], description="Hora")
importe = widgets.FloatText(value=compra_online[2], description="USD")
modalidad = widgets.Dropdown(options=MODALIDADES, value=compra_online[3], description="Modalidad")
estado = widgets.Text(value=compra_online[4], description="Estado")
mcc = widgets.IntText(value=compra_online[5], description="MCC")
evaluar = widgets.Button(description="Evaluar transacción", button_style="primary")
salida = widgets.Output()
campos = [fecha, hora, importe, modalidad, estado, mcc]

def on_evaluar(_):
    with salida:
        salida.clear_output(wait=True)
        try:
            display(mostrar(modelo.value, [("Consulta", [c.value for c in campos])]))
        except (TypeError, ValueError) as error:
            print(f"Revisa los datos ingresados. {error}")

evaluar.on_click(on_evaluar)
display(widgets.VBox([modelo, *campos, evaluar, salida]))

## 5. Demo web

Desde la raíz del repositorio, ejecuta `python -m unidad3.app` y abre [http://127.0.0.1:7860](http://127.0.0.1:7860). El archivo `unidad3/README.md` documenta la instalación y el reentrenamiento.